# Amazon Bedrock: Converse API & Tool Calling

*This notebook walks through building conversational agents and integrating external tools (function calling with AWS Lambda) using the **Amazon Bedrock Converse API** and **Amazon Nova models**. Adapted from the `plan.md` exercise guide.*

---

## Architecture: How Tool Calling Works in Bedrock

Large Language Models (LLMs) excel at reasoning and language generation, but cannot natively perform precise arithmetic, query dynamic databases, or interact with external systems. **Tool Use (Function Calling)** bridges this gap:

### 1. Sequence Flow Diagram

<div align="center">
  <img src="../images/tool_calling_sequence.svg" alt="Amazon Bedrock Tool Calling Sequence Diagram" width="100%" />
</div>

<br/>

### 2. Component Architecture

<div align="center">
  <img src="../images/tool_calling_architecture.svg" alt="Amazon Bedrock Tool Calling Component Architecture" width="100%" />
</div>

---

### Notebook Workflow
1. **Task 1 & 2**: Multi-turn conversation with system prompts using the Bedrock Converse API.
2. **Task 3**: Building and invoking a CHAT prompt template in Bedrock Prompt Management.
3. **Task 4**: Implementing the arithmetic calculation function for AWS Lambda.
4. **Task 5**: Defining the tool schema (`toolConfig`) and orchestrating the end-to-end tool calling loop.
5. **Task 6**: Resource cleanup and lifecycle management.


## Task 1 & 2.1: Initialize Bedrock and Set Defaults

We initialize three Boto3 clients:
- `bedrock-runtime`: For invoking models with `converse` and `converse_stream`.
- `bedrock-agent`: For managing and deploying Bedrock Prompts.
- `lambda`: For invoking external AWS Lambda functions.

> [!IMPORTANT]
> **Region & Model Configuration**:
> - **Target Region**: `us-east-2`
> - **Model ID**: `us.amazon.nova-lite-v1:0` (System Inference Profile with `us.` prefix required for on-demand Bedrock invocations).

In [1]:
import boto3
import json
import datetime
from botocore.config import Config
from botocore.exceptions import ClientError

# 1. Configure target region
REGION = "us-east-2"

client_config = Config(
    region_name=REGION,
    connect_timeout=60,
    read_timeout=120,
    retries={"max_attempts": 3, "mode": "standard"}
)

# 2. Initialize Bedrock & Lambda clients
bedrock = boto3.client(service_name="bedrock-runtime", config=client_config)
bedrock_agent = boto3.client(service_name="bedrock-agent", config=client_config)
lambda_client = boto3.client(service_name="lambda", config=client_config)

# 3. Model identifier with required 'us.' prefix for Cross-Region Inference
NOVA_LITE = "us.amazon.nova-lite-v1:0"
NOVA_MICRO = "us.amazon.nova-micro-v1:0"

MODEL_ID = NOVA_LITE

print(f"Bedrock clients successfully initialized for region: {REGION}")
print(f"Target Foundation Model: {MODEL_ID}")


Bedrock clients successfully initialized for region: us-east-2
Target Foundation Model: us.amazon.nova-lite-v1:0


## Task 2: Multi-Turn Conversing with Bedrock

In this task, we build a multi-turn conversation with Bedrock as a **Virtual Travel Assistant**.

### Task 2.2: Initial Travel Request
We supply:
1. A **System Prompt** defining the persona and constraints: *"You are a virtual travel assistant that suggests destinations based on user preferences. Only return destination names and a brief description."*
2. An **Inference Configuration** with `temperature = 0.7`.
3. The initial user request (`message_1`).

In [2]:
# Configure inference parameters
temperature = 0.7
inference_config = {"temperature": temperature, "maxTokens": 500}

# System prompt establishing the persona
system_prompts = [{
    "text": "You are a virtual travel assistant that suggests destinations based on user preferences. "
            "Only return destination names and a brief description."
}]

# Initialize message history list
messages = []

# Turn 1: User's initial destination query
message_1 = {
    "role": "user",
    "content": [{"text": "I want to plan a 4-day vacation somewhere warm with beautiful beaches and great seafood."}]
}
messages.append(message_1)

print("=== User Turn 1 ===")
print(message_1["content"][0]["text"])

# Call Bedrock Converse API
response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

# Extract assistant response
assistant_response_1 = response["output"]["message"]
messages.append(assistant_response_1)

print("\n=== Assistant Response 1 ===")
print(assistant_response_1["content"][0]["text"])


=== User Turn 1 ===
I want to plan a 4-day vacation somewhere warm with beautiful beaches and great seafood.

=== Assistant Response 1 ===
### Maldives
**Description:** A tropical paradise with crystal-clear waters, white-sand beaches, and vibrant coral reefs. Enjoy delicious seafood at local restaurants while relaxing on the shore.


### Task 2.3: Continuing the Conversation (Refinement)

We continue the conversation by appending a second user message (`message_2`) that narrows down the recommendations: *"Only suggest travel locations that are no more than one short flight away from Miami, Florida."*

Because the Converse API preserves the full conversation array in `messages`, the model contextualizes the new constraint against the previous beach destinations.

In [3]:
# Turn 2: Add flight distance constraint
message_2 = {
    "role": "user",
    "content": [{"text": "Only suggest travel locations that are no more than one short flight away from Miami, Florida."}]
}
messages.append(message_2)

print("=== User Turn 2 ===")
print(message_2["content"][0]["text"])

# Converse call with accumulated conversation history
response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

assistant_response_2 = response["output"]["message"]
messages.append(assistant_response_2)

print("\n=== Assistant Response 2 ===")
print(assistant_response_2["content"][0]["text"])


=== User Turn 2 ===
Only suggest travel locations that are no more than one short flight away from Miami, Florida.

=== Assistant Response 2 ===
### Jamaica
**Description:** Experience the warm Caribbean sun with Jamaica's stunning beaches and crystal-clear waters. Indulge in fresh seafood at local eateries while enjoying reggae beats and lush landscapes.


### Task 2.4: Further Refining the Recommendations

Now add a third turn asking Bedrock to sort the destinations by proximity and estimate typical flight durations:

In [4]:
# Turn 3: Request proximity sorting and flight times
message_3 = {
    "role": "user",
    "content": [{"text": "List the suggested destinations in order of proximity to Miami, FL and estimate the approximate flight duration for each."}]
}
messages.append(message_3)

print("=== User Turn 3 ===")
print(message_3["content"][0]["text"])

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

assistant_response_3 = response["output"]["message"]
messages.append(assistant_response_3)

print("\n=== Assistant Response 3 ===")
print(assistant_response_3["content"][0]["text"])
print("\nTotal turns in conversation history:", len(messages))


=== User Turn 3 ===
List the suggested destinations in order of proximity to Miami, FL and estimate the approximate flight duration for each.

=== Assistant Response 3 ===
### Bahamas (Nassau)
**Description:** Close to Miami, with stunning beaches and clear waters. Enjoy delicious seafood and vibrant local culture. 
**Estimated Flight Duration:** ~1 hour

### Dominican Republic (Punta Cana)
**Description:** Known for its beautiful beaches and luxury resorts. Experience delicious Caribbean cuisine and water sports.
**Estimated Flight Duration:** ~1.5 hours

### Jamaica (Montego Bay)
**Description:** Enjoy Jamaica's warm climate, beautiful beaches, and delicious seafood.
**Estimated Flight Duration:** ~2 hours

Total turns in conversation history: 6


## Task 3: Creating a Bedrock Prompt (Chat Template)

In this task, we create a reusable prompt template in **Amazon Bedrock Prompt Management** designed to evaluate trip planning requests like a travel agent would.

- **Template Type**: `CHAT`
- **Input Variable**: `{{event_request}}`
- **Model**: `us.amazon.nova-lite-v1:0`
- **Evaluation Criteria**: Checks if the request contains 1) Destination, 2) Dates/duration, 3) Party size, 4) Budget/travel style, and 5) Specific activities.

In [5]:
# Unique timestamped prompt name for safe re-runs
prompt_name = f"Travel-Agent-Prompt-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"

system_instruction = (
    "You are an expert travel consultant. Your job is to thoroughly evaluate customer trip planning inquiries.\n"
    "Evaluate whether the request provides:\n"
    "1. Specific destination or region\n"
    "2. Travel dates and duration\n"
    "3. Travel party size and composition\n"
    "4. Budget tier or travel style\n"
    "5. Preferred activities or restrictions\n\n"
    "Provide a concise assessment of what information is present, what is missing, and 3 specific follow-up questions to ask the customer."
)

print(f"Creating Bedrock CHAT prompt: {prompt_name}...")
prompt_resp = bedrock_agent.create_prompt(
    name=prompt_name,
    description="Checks if all trip planning information has been provided.",
    defaultVariant="Variant1",
    variants=[
        {
            "name": "Variant1",
            "modelId": MODEL_ID,
            "templateType": "CHAT",
            "inferenceConfiguration": {
                "text": {
                    "temperature": 0.4,
                    "topP": 0.9,
                    "maxTokens": 1000
                }
            },
            "templateConfiguration": {
                "chat": {
                    "inputVariables": [
                        {"name": "event_request"}
                    ],
                    "system": [
                        {"text": system_instruction}
                    ],
                    "messages": [
                        {
                            "role": "user",
                            "content": [
                                {"text": "Please evaluate this trip planning request:\n\n{{event_request}}"}
                            ]
                        }
                    ]
                }
            }
        }
    ]
)

prompt_id = prompt_resp["id"]
prompt_arn = prompt_resp["arn"]
print(f"Created Prompt: {prompt_resp['name']}")
print(f"Prompt ID:     {prompt_id}")
print(f"Prompt ARN:    {prompt_arn}")

# Publish Version 1 of this prompt
version_resp = bedrock_agent.create_prompt_version(
    description="Version 1 of Travel Agent evaluation prompt.",
    promptIdentifier=prompt_arn
)
prompt_version_arn = version_resp["arn"]
print(f"Published Version 1 ARN: {prompt_version_arn}")


Creating Bedrock CHAT prompt: Travel-Agent-Prompt-20261116212423...
Created Prompt: Travel-Agent-Prompt-20261116212423
Prompt ID:     1CSFEXP2XP
Prompt ARN:    arn:aws:bedrock:us-east-2:994770697278:prompt/1CSFEXP2XP
Published Version 1 ARN: arn:aws:bedrock:us-east-2:994770697278:prompt/1CSFEXP2XP:1


In [6]:
# Sample trip planning inquiry
sample_trip_request = """
Hi there! I'm planning a trip to Italy with my partner and would love some help organizing the itinerary. 
We're hoping to travel between September 10–20 this year, ideally flying into Rome and spending a few days 
in Florence and Venice as well. We'd love recommendations on tours, cultural sites, and good local restaurants. 
We're not interested in anything rushed.
"""

print("Invoking Prompt Version with customer inquiry...")
eval_response = bedrock.converse(
    modelId=prompt_version_arn,
    promptVariables={
        "event_request": {
            "text": sample_trip_request.strip()
        }
    }
)

print("\n=== Travel Consultant Prompt Assessment ===\n")
print(eval_response["output"]["message"]["content"][0]["text"])


Invoking Prompt Version with customer inquiry...

=== Travel Consultant Prompt Assessment ===

### Evaluation of the Trip Planning Request

**Present Information:**
1. **Specific Destination or Region:** Yes, Italy.
2. **Travel Dates and Duration:** Yes, September 10–20, 2023 (11 days).
3. **Travel Party Size and Composition:** Yes, the customer and their partner (2 people).
4. **Preferred Activities or Restrictions:** Yes, recommendations on tours, cultural sites, and good local restaurants. They prefer a leisurely pace and are not interested in anything rushed.

**Missing Information:**
1. **Budget Tier or Travel Style:** The customer has not specified their budget or preferred travel style (e.g., luxury, budget, mid-range).

### Follow-Up Questions:

1. **Budget and Travel Style:**
   - "To better tailor your itinerary, could you please provide an approximate budget for your trip per person per day, as well as your preferred travel style (e.g., luxury, mid-range, budget)?"

2. **Acc

## Task 4: AWS Lambda Function (Math Operations)

LLMs frequently struggle with multi-step arithmetic. In this task, we build an external calculation tool.

### Lambda Function Implementation
The following Python code represents the AWS Lambda function (`math-function`):

```python
import json

def add(num1, num2): return num1 + num2
def subtract(num1, num2): return num1 - num2
def multiply(num1, num2): return num1 * num2
def divide(num1, num2):
    if num2 == 0:
        raise ValueError("Division by zero is not allowed")
    return num1 / num2

def lambda_handler(event, context):
    operation = event.get('operation')
    num1 = float(event.get('num1', 0))
    num2 = float(event.get('num2', 0))
    
    if operation == 'add': result = add(num1, num2)
    elif operation == 'subtract': result = subtract(num1, num2)
    elif operation == 'multiply': result = multiply(num1, num2)
    elif operation == 'divide': result = divide(num1, num2)
    else: return {'statusCode': 400, 'body': json.dumps({'error': f'Unsupported operation: {operation}'})}
    
    return {'statusCode': 200, 'body': json.dumps({'result': result})}
```

> [!TIP]
> The cell below provides a **dual-mode dispatcher**: If you deployed `math-function` in AWS Lambda, it calls AWS Lambda via `boto3`. If not deployed yet, it executes the identical Python logic locally, ensuring your notebook runs end-to-end without blockers.

In [7]:
# Core arithmetic logic
def add(num1, num2):
    return num1 + num2

def subtract(num1, num2):
    return num1 - num2

def multiply(num1, num2):
    return num1 * num2

def divide(num1, num2):
    if num2 == 0:
        raise ValueError("Division by zero is not allowed.")
    return num1 / num2

def execute_math_tool(operation: str, num1: float, num2: float, lambda_func_name: str = "math-function"):
    """
    Attempts to invoke the deployed AWS Lambda function 'math-function'.
    Falls back seamlessly to local execution if the Lambda function is not deployed.
    """
    payload = {"operation": operation, "num1": num1, "num2": num2}
    
    # Attempt live AWS Lambda invocation
    try:
        resp = lambda_client.invoke(
            FunctionName=lambda_func_name,
            InvocationType="RequestResponse",
            Payload=json.dumps(payload)
        )
        raw_result = json.loads(resp["Payload"].read().decode("utf-8"))
        if "body" in raw_result:
            body_data = json.loads(raw_result["body"]) if isinstance(raw_result["body"], str) else raw_result["body"]
            print(f"[AWS Lambda '{lambda_func_name}'] Output: {body_data}")
            return body_data
        print(f"[AWS Lambda '{lambda_func_name}'] Output: {raw_result}")
        return raw_result
    except (ClientError, Exception) as e:
        # Fallback to local python calculation
        print(f"[Local Dispatcher Fallback] (Note: Lambda '{lambda_func_name}' not reachable: {e})")
        try:
            if operation == "add":
                res = add(num1, num2)
            elif operation == "subtract":
                res = subtract(num1, num2)
            elif operation == "multiply":
                res = multiply(num1, num2)
            elif operation == "divide":
                res = divide(num1, num2)
            else:
                return {"error": f"Unknown operation: {operation}"}
            
            local_output = {"result": float(res)}
            print(f"Tool Output: {local_output}")
            return local_output
        except Exception as local_err:
            err_output = {"error": str(local_err)}
            print(f"Tool Error Output: {err_output}")
            return err_output

# Test local dispatcher
test_sample = execute_math_tool("add", 10, 5)
print("Sample Test (10 + 5):", test_sample)


[Local Dispatcher Fallback] (Note: Lambda 'math-function' not reachable: An error occurred (ResourceNotFoundException) when calling the Invoke operation: Function not found: arn:aws:lambda:us-east-2:994770697278:function:math-function:$LATEST)
Tool Output: {'result': 15.0}
Sample Test (10 + 5): {'result': 15.0}


## Task 5: Integrating Bedrock and Lambda with `toolConfig`

Now we tell Amazon Bedrock about our calculation tool by passing a **`toolConfig`** specification to the Converse API:

- **`name`**: `calculateNumbers`
- **`description`**: Explains what the tool does and when the model should invoke it.
- **`inputSchema`**: JSON Schema defining the required input properties (`operation`, `num1`, `num2`).

In [8]:
# Define calculation tool specification for Bedrock Converse API
math_tool = {
    "toolSpec": {
        "name": "calculateNumbers",
        "description": "Performs mathematical calculations including addition, subtraction, multiplication, and division.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["add", "subtract", "multiply", "divide"],
                        "description": "The arithmetic operation to perform."
                    },
                    "num1": {
                        "type": "number",
                        "description": "The first number (left operand)."
                    },
                    "num2": {
                        "type": "number",
                        "description": "The second number (right operand)."
                    }
                },
                "required": ["operation", "num1", "num2"]
            }
        }
    }
}

tool_config = {
    "tools": [math_tool]
}

print("Tool configuration registered:")
print(json.dumps(tool_config, indent=2))


Tool configuration registered:
{
  "tools": [
    {
      "toolSpec": {
        "name": "calculateNumbers",
        "description": "Performs mathematical calculations including addition, subtraction, multiplication, and division.",
        "inputSchema": {
          "json": {
            "type": "object",
            "properties": {
              "operation": {
                "type": "string",
                "enum": [
                  "add",
                  "subtract",
                  "multiply",
                  "divide"
                ],
                "description": "The arithmetic operation to perform."
              },
              "num1": {
                "type": "number",
                "description": "The first number (left operand)."
              },
              "num2": {
                "type": "number",
                "description": "The second number (right operand)."
              }
            },
            "required": [
              "operation",
       

### Running the Tool Calling Loop

When the user asks: *"What is 100 minus 60?"*:
1. Bedrock evaluates the prompt and determines that `calculateNumbers` is needed.
2. Bedrock returns `stopReason: 'tool_use'` with `toolUseId` and parameters (`num1: 100, operation: 'subtract', num2: 60`).
3. We execute the tool using our function.
4. We send back a message with `role: 'user'` and a `toolResult` content block.
5. Bedrock synthesizes the tool result into a natural language answer.

In [9]:
# Query requiring exact math
math_messages = [{
    "role": "user",
    "content": [{"text": "What is 100 minus 60?"}]
}]

print("=== 1. Initial User Request ===")
print(math_messages[0]["content"][0]["text"])

# First Converse call
response = bedrock.converse(
    modelId=MODEL_ID,
    messages=math_messages,
    toolConfig=tool_config,
    inferenceConfig={"temperature": 0.0, "maxTokens": 500}
)

stop_reason = response.get("stopReason")
print(f"\nBedrock Stop Reason: {stop_reason}")

assistant_msg = response["output"]["message"]
math_messages.append(assistant_msg)

# Check if Bedrock requested tool execution
if stop_reason == "tool_use":
    for content_block in assistant_msg.get("content", []):
        if "toolUse" in content_block:
            tool_use = content_block["toolUse"]
            tool_use_id = tool_use["toolUseId"]
            tool_name = tool_use["name"]
            tool_input = tool_use["input"]
            
            print(f"\n=== 2. Assistant Requested Tool Call ===")
            print(f"Tool Name:   {tool_name}")
            print(f"Tool Use ID: {tool_use_id}")
            print(f"Tool Input:  {tool_input}")
            
            # Execute the calculation
            result_data = execute_math_tool(
                operation=tool_input["operation"],
                num1=tool_input["num1"],
                num2=tool_input["num2"]
            )
            
            # Format tool result block
            tool_result_message = {
                "role": "user",
                "content": [
                    {
                        "toolResult": {
                            "toolUseId": tool_use_id,
                            "content": [{"json": result_data}],
                            "status": "success" if "error" not in result_data else "error"
                        }
                    }
                ]
            }
            math_messages.append(tool_result_message)

    # Second Converse call returning the tool result to Bedrock
    print("\n=== 3. Sending Tool Result Back to Bedrock ===")
    followup_resp = bedrock.converse(
        modelId=MODEL_ID,
        messages=math_messages,
        toolConfig=tool_config,
        inferenceConfig={"temperature": 0.0, "maxTokens": 500}
    )
    
    final_answer = followup_resp["output"]["message"]["content"][0]["text"]
    print("\n=== Final Assistant Response ===")
    print(final_answer)
else:
    print("Assistant responded directly without calling a tool:")
    print(assistant_msg["content"][0]["text"])


=== 1. Initial User Request ===
What is 100 minus 60?

Bedrock Stop Reason: tool_use

=== 2. Assistant Requested Tool Call ===
Tool Name:   calculateNumbers
Tool Use ID: tooluse_HLs1V5tNgptRxafVS6qz0d
Tool Input:  {'num1': 100, 'operation': 'subtract', 'num2': 60}
[Local Dispatcher Fallback] (Note: Lambda 'math-function' not reachable: An error occurred (ResourceNotFoundException) when calling the Invoke operation: Function not found: arn:aws:lambda:us-east-2:994770697278:function:math-function:$LATEST)
Tool Output: {'result': 40.0}

=== 3. Sending Tool Result Back to Bedrock ===

=== Final Assistant Response ===
The result of 100 minus 60 is 40.


### Testing Variations: Word Problems and Error Handling

Let's test two critical scenarios:
1. **Word Problem / Real-World Calculation**: *"A customer orders 24 units at $37.50 each. What is the total cost?"*
2. **Error Handling (Division by Zero)**: *"What is 42 divided by 0?"* (Inspect how the model handles a tool error message).

In [10]:
def run_tool_conversation(query: str, verbose: bool = True):
    """
    Reusable orchestrator function that handles the complete Bedrock tool-calling lifecycle.
    """
    convo = [{"role": "user", "content": [{"text": query}]}]
    if verbose:
        print(f"\nUser Query: '{query}'")
        
    # Initial call
    resp = bedrock.converse(
        modelId=MODEL_ID,
        messages=convo,
        toolConfig=tool_config,
        inferenceConfig={"temperature": 0.0, "maxTokens": 400}
    )
    
    assistant_msg = resp["output"]["message"]
    convo.append(assistant_msg)
    
    # Tool handling loop
    if resp.get("stopReason") == "tool_use":
        for block in assistant_msg.get("content", []):
            if "toolUse" in block:
                t_use = block["toolUse"]
                if verbose:
                    print(f"-> Triggered Tool: {t_use['name']} with {t_use['input']}")
                
                # Execute
                res = execute_math_tool(
                    operation=t_use["input"]["operation"],
                    num1=t_use["input"]["num1"],
                    num2=t_use["input"]["num2"]
                )
                
                status = "error" if "error" in res else "success"
                convo.append({
                    "role": "user",
                    "content": [{
                        "toolResult": {
                            "toolUseId": t_use["toolUseId"],
                            "content": [{"json": res}],
                            "status": status
                        }
                    }]
                })
                
        # Follow-up call
        final_resp = bedrock.converse(
            modelId=MODEL_ID,
            messages=convo,
            toolConfig=tool_config,
            inferenceConfig={"temperature": 0.0, "maxTokens": 400}
        )
        final_text = final_resp["output"]["message"]["content"][0]["text"]
    else:
        final_text = assistant_msg["content"][0]["text"]
        
    if verbose:
        print(f"Assistant Answer:\n{final_text}\n")
    return final_text

print("=== Test 1: Word Problem (Multiplication) ===")
run_tool_conversation("A customer orders 24 units at $37.50 each. What is the total cost?")

print("=" * 60)
print("=== Test 2: Division by Zero (Tool Error Handling) ===")
run_tool_conversation("Can you divide 42 by 0?")


=== Test 1: Word Problem (Multiplication) ===

User Query: 'A customer orders 24 units at $37.50 each. What is the total cost?'
-> Triggered Tool: calculateNumbers with {'num1': 24, 'operation': 'multiply', 'num2': 37.5}
[Local Dispatcher Fallback] (Note: Lambda 'math-function' not reachable: An error occurred (ResourceNotFoundException) when calling the Invoke operation: Function not found: arn:aws:lambda:us-east-2:994770697278:function:math-function:$LATEST)
Tool Output: {'result': 900.0}
Assistant Answer:
<thinking>I have used the 'calculateNumbers' tool to multiply the number of units (24) by the price per unit ($37.50). The result is $900, which is the total cost of the order.</thinking>

The total cost of the order is $900.

=== Test 2: Division by Zero (Tool Error Handling) ===

User Query: 'Can you divide 42 by 0?'
Assistant Answer:
<thinking>The User is asking to divide 42 by 0. Division by zero is undefined in mathematics. I should inform the User that this operation is not p

"<thinking>The User is asking to divide 42 by 0. Division by zero is undefined in mathematics. I should inform the User that this operation is not possible.</thinking>\n\n<div>I'm sorry, but division by zero is undefined in mathematics. It's not possible to divide any number by zero.</div>"

## Task 6: Clean Up & Resource Management

### Resource Deletion
1. **Bedrock Prompt**: We delete the `Travel-Agent-Prompt` created in Task 3 to keep our account clean.
2. **AWS Lambda**: If you deployed the `math-function` Lambda in the AWS console, you can delete it from the AWS Lambda dashboard.
3. **Zero Idle Invocations**: On-demand Bedrock `converse` calls incur no persistent hourly charge.

In [11]:
# Safely delete the prompt created in Task 3
try:
    print(f"Deleting exercise prompt ID: {prompt_id} ({prompt_name})...")
    bedrock_agent.delete_prompt(promptIdentifier=prompt_id)
    print("Exercise prompt successfully deleted!")
except Exception as e:
    print(f"Prompt cleanup note: {e}")

# Optional: Delete AWS Lambda function if created programmatically or verify status
try:
    lambda_info = lambda_client.get_function(FunctionName="math-function")
    print(f"\nDiscovered AWS Lambda function 'math-function' (ARN: {lambda_info['Configuration']['FunctionArn']})")
    print("To delete this Lambda function, run: lambda_client.delete_function(FunctionName='math-function')")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print("\nAWS Lambda 'math-function' is not present in AWS account (local dispatcher was used). Zero cleanup required.")
    else:
        print(f"Lambda check note: {e}")


Deleting exercise prompt ID: 1CSFEXP2XP (Travel-Agent-Prompt-20261116212423)...
Exercise prompt successfully deleted!

AWS Lambda 'math-function' is not present in AWS account (local dispatcher was used). Zero cleanup required.


## Summary

In this notebook, you learned:
- ✅ **Converse API Multi-turn**: Managed ongoing chat sessions and conversational memory using structured message arrays.
- ✅ **Chat Prompt Management**: Created and published a versioned `CHAT` template (`Travel-Agent-Prompt`) in Amazon Bedrock with dynamic variables.
- ✅ **Tool Specification (`toolConfig`)**: Defined JSON schema specifications enabling Amazon Nova models to detect when external functions are required.
- ✅ **Tool Execution Cycle**: Intercepted `stopReason: 'tool_use'`, dispatched execution to an arithmetic handler / AWS Lambda, and returned structured `toolResult` blocks.
- ✅ **Error Handling in Tools**: Gracefully managed runtime tool errors (e.g., division by zero) and observed the model synthesize friendly feedback.

### Complete Bedrock Exploration Series
- [001-Invoke-model.ipynb](001-Invoke-model.ipynb): Model discovery, synchronous invocations, prompt engineering.
- [001b-Converse-API.ipynb](001b-Converse-API.ipynb): Universal Converse API, streaming, and multimodal image/document analysis.
- [002-Streaming-response.ipynb](002-Streaming-response.ipynb): Streaming token lifecycle, TTFT metrics, and generator patterns.
- [003-Guardrails.ipynb](003-Guardrails.ipynb): Content moderation, PII redaction, topic filtering, and programmatic Guardrail APIs.
- [004-Knowledge-Base.ipynb](004-Knowledge-Base.ipynb): Retrieval-Augmented Generation (RAG) and Titan Text Embeddings V2.
- [005-Prompt-Management.ipynb](005-Prompt-Management.ipynb): Reusable prompt templates, versioning, and prompt optimization.
- [006-Tool-Calling.ipynb](006-Tool-Calling.ipynb): Function calling, tool specifications, and AWS Lambda integration.
